# Notes Spark Batch Processing - Module 6 DE Zoomcamp

## PARTIE 1 : APACHE SPARK

### Qu'est-ce que Spark ?

**Apache Spark** = Moteur de traitement de données distribué qui répartit le calcul sur plusieurs machines (ou cœurs) en parallèle pour traiter des volumes massifs de données.

**PySpark** = API (Application Programming Interface) Python de Spark. Le moteur tourne en Scala/Java sous le capot, mais tu écris ton code en Python.

**Le problème que Spark résout :**
- **pandas** : charge tout en RAM → plante sur 500 Go
- **PostgreSQL** : un seul serveur → lent, ne scale pas
- **Spark** : découpe les données en morceaux, distribue sur N workers, agrège les résultats

**Ce que Spark N'EST PAS :**
- Pas un système de stockage (pas une base de données)
- Pas un outil OLTP (Online Transaction Processing)
- Pas idéal pour les petits datasets (pandas est plus rapide sous ~1 Go)
- Pas un outil de monitoring

---

### Batch vs Streaming

**BATCH** = Tu traites un jeu de données fini, déjà stocké (fichiers Parquet, CSV, data lake...). → Module 6

**STREAMING** = Tu traites des données en continu, au fil de l'eau (Kafka...). → Module 7

```
BATCH    : fichier complet → Spark → résultat final
STREAMING: flux continu → Spark → résultats en temps réel
```

---

### Architecture Spark

**3 composants :**

```
Driver (ton programme PySpark)
    ↓  envoie le plan d'exécution
Cluster Manager (gère les ressources)
    ↓  distribue le travail
Worker 1 | Worker 2 | Worker 3  (font le calcul en parallèle)
```

- **Driver** : ton script PySpark. Il orchestre, il ne calcule pas.
- **Executor** : les workers qui font le vrai calcul. Chacun a des cœurs CPU et de la RAM alloués.
- **Cluster Manager** : YARN, Kubernetes, ou Spark Standalone. Il attribue les ressources.

**En local (`local[*]`) :** tout tourne sur ta machine avec plusieurs threads. C'est le mode utilisé dans le Zoomcamp.

---

### DataFrame Spark vs pandas

| pandas DataFrame | Spark DataFrame |
|---|---|
| En mémoire RAM, sur une machine | Distribué sur N workers |
| Exécution immédiate | Exécution **lazy** (voir ci-dessous) |
| Limité à la RAM disponible | Scale jusqu'à des To (téraoctets) |
| `import pandas as pd` | `from pyspark.sql import SparkSession` |

**RDD (Resilient Distributed Dataset)** = La primitive de base de Spark. Une collection distribuée, tolérante aux pannes. Bas niveau — tu n'y touches presque plus directement. Les DataFrames sont construits par-dessus.

---

### Concept clé : Lazy Evaluation

**Rien ne s'exécute immédiatement.** Spark construit un plan logique (le DAG = Directed Acyclic Graph, graphe orienté sans cycle) et attend une **action** pour déclencher l'exécution.

```python
df = spark.read.parquet("dvf.parquet")           # rien ne se passe
df2 = df.filter(df.code_dept == "974")           # rien ne se passe
df3 = df2.groupBy("commune").count()             # rien ne se passe

df3.show()  # ← ICI Spark exécute TOUT d'un coup
```

**Pourquoi ?** Spark optimise le plan global avant de lancer quoi que ce soit via le **Catalyst Optimizer**. Il réorganise les opérations, élimine l'inutile, choisit les meilleures jointures.

**2 types d'opérations :**

| Type | Exemples | Effet |
|---|---|---|
| **Transformations** (lazy) | `filter`, `select`, `groupBy`, `join`, `withColumn` | Construisent le plan, rien n'est exécuté |
| **Actions** (déclenchent l'exécution) | `show()`, `count()`, `collect()`, `write()` | Spark exécute le DAG |

---

### Partitions

Spark ne manipule jamais le dataset entier d'un coup. Il le découpe en **partitions** — des blocs de données indépendants traités en parallèle par les executors.

```
Dataset 500 Go
→ Partition 1 (25 Go) → Executor 1
→ Partition 2 (25 Go) → Executor 2
→ ...
→ Partition 20 (25 Go) → Executor 20
```

**Règle pratique :** 1 partition par cœur CPU disponible, entre 100 Mo et 200 Mo par partition.

**Commandes :**
```python
df.repartition(4)    # Augmente ou réduit les partitions (shuffle)
df.coalesce(2)       # Réduit seulement les partitions (pas de shuffle)
```

---

### Le Shuffle : l'opération coûteuse

Certaines opérations nécessitent de **déplacer des données entre workers** — c'est le **shuffle**. C'est la chose la plus coûteuse en Spark.

**Déclenché par :** `groupBy`, `join`, `distinct`, `orderBy`, `repartition`

```
Avant shuffle : chaque worker a ses données locales
Après shuffle  : les données sont redistribuées pour que
                 les mêmes clés soient sur le même worker
```

**Règle d'or :** quand tu optimises du code Spark, la première question est toujours — *est-ce que je déclenche des shuffles inutiles ?*

---

## PARTIE 2 : PYSPARK EN PRATIQUE

### Installation (WSL2)

**Prérequis : Java (obligatoire)**
```bash
sudo apt update && sudo apt install -y openjdk-17-jdk
java -version  # → openjdk version "17.x.x"
```

**Installer PySpark via uv :**
```bash
uv venv .venv
source .venv/bin/activate
uv pip install pyspark
# ⚠️ PySpark pèse ~300 Mo, c'est normal que ça prenne du temps
```

**Variables d'environnement à garder dans `.zshrc` :**
```bash
export JAVA_HOME="/usr/lib/jvm/java-17-openjdk-amd64"
export PYSPARK_PYTHON="python3"
# ⚠️ Ne PAS mettre SPARK_HOME si tu utilises PySpark via pip/uv
#    PySpark installé via uv est standalone — il embarque Spark lui-même
```

**Piège classique :** Si tu avais `SPARK_HOME=/opt/spark` dans ton `.zshrc` sans avoir installé Spark manuellement dans `/opt/spark`, PySpark ne trouve pas `spark-submit` et plante.
```bash
# Fix : désactiver SPARK_HOME avant de lancer
unset SPARK_HOME
```

---

### SparkSession — Le Point d'Entrée

```python
from pyspark.sql import SparkSession

# Toujours la première étape — une seule session par programme
spark = SparkSession.builder \
    .master("local[*]") \   # local[*] = tous les cœurs dispo
    .appName("MonApp") \
    .getOrCreate()          # getOrCreate = réutilise si existe déjà

# Vérifier la version
print(spark.version)  # → 4.1.1

# Toujours fermer à la fin
spark.stop()
```

---

### Workflow Complet

```
Source de données
(Parquet, CSV, S3, GCS...)
        ↓
1. INGEST — Lire les données
   spark.read.parquet(...)
        ↓
2. EXPLORE — Comprendre le dataset
   df.printSchema() / df.show() / df.describe()
        ↓
3. TRANSFORM — Nettoyer, enrichir, filtrer
   .filter() / .withColumn() / .join() / .groupBy()
        ↓
4. PARTITION — Optimiser le stockage
   .repartition(N)
        ↓
5. SINK — Écrire le résultat
   .write.parquet(...) / .write.bigquery(...)
```

---

### Opérations Essentielles

**Lire des données :**
```python
df = spark.read.parquet("fichier.parquet")
df = spark.read.option("header", "true").csv("fichier.csv")
df = spark.read.json("fichier.json")
```

**Explorer :**
```python
df.printSchema()          # Structure des colonnes et types
df.show(5)                # Afficher les 5 premières lignes
df.describe().show()      # Statistiques descriptives
df.count()                # Nombre de lignes (action !)
```

**Transformer :**
```python
from pyspark.sql.functions import col, to_date, unix_timestamp

# Filtrer
df.filter(col("code_dept") == "974")
df.filter(to_date(col("pickup_datetime")) == "2025-11-15")

# Ajouter une colonne calculée
df.withColumn(
    "duration_hours",
    (unix_timestamp("dropoff_datetime") - unix_timestamp("pickup_datetime")) / 3600
)

# Grouper et agréger
df.groupBy("commune").count()
df.groupBy("zone").agg({"revenue": "sum"})
df.agg({"duration_hours": "max"})
```

**Temp Views (SQL sur Spark) :**
```python
# Enregistrer un DataFrame comme table temporaire
df.createOrReplaceTempView("trips")
zones.createOrReplaceTempView("zones")

# Requêter en SQL pur
spark.sql("""
    SELECT z.Zone, COUNT(*) as cnt
    FROM trips t
    JOIN zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Zone
    ORDER BY cnt ASC
    LIMIT 1
""").show()
```

**Écrire :**
```python
df.write.parquet("output/")
df.repartition(4).write.parquet("output/partitionné/")
df.write.mode("overwrite").parquet("output/")
```

---

### Collect — Attention

```python
# ⚠️ collect() ramène TOUT en mémoire driver
# OK uniquement pour de petits résultats
result = df.agg({"col": "max"}).collect()[0][0]  # 1 seule valeur → safe
df.collect()  # ❌ JAMAIS sur un gros dataset → plante
```

---

## PARTIE 3 : SPARK UI

**Port par défaut : 4040**

Accessible sur `http://localhost:4040` pendant l'exécution d'un job Spark.

**Ce qu'on y trouve :**
- **Jobs** : chaque action déclenchée (show, count, write...)
- **Stages** : les étapes d'exécution d'un job
- **Tasks** : le travail sur chaque partition
- **Storage** : DataFrames mis en cache
- **Environment** : config Spark active

**Utilité :** Détecter les shuffles coûteux, les skews de données (partitions déséquilibrées), les stages lents.

---

## PARTIE 4 : FORMATS DE FICHIERS

### Parquet — Le Format Spark

**Parquet** = Format de stockage **colonnaire**, compressé, optimisé pour la lecture analytique.

| Format | Usage | Avantages |
|---|---|---|
| **Parquet** | Analytics, data lake | Columnar, compressé, rapide à lire |
| CSV | Échange, lisibilité humaine | Simple, universel |
| JSON | APIs, semi-structuré | Flexible, lisible |

**Pourquoi Parquet > CSV en Spark :**
- Lit uniquement les colonnes nécessaires (columnar)
- Compression native (~5x moins lourd qu'un CSV équivalent)
- Les types de données sont préservés (pas de parsing)

**Repartitionner avant d'écrire :**
```python
# 4 fichiers Parquet de taille équilibrée
df.repartition(4).write.parquet("output/")

# Vérifier les tailles après écriture
# ls -lh output/*.parquet
```

---

## Ce que Spark fait vs ne fait pas

| ✅ Fait bien | ❌ Ne fait pas / ne remplace pas |
|---|---|
| Transformations ETL massives (Go/To) | Système de stockage (pas un data lake) |
| Lecture/écriture Parquet, CSV, JSON, Delta | Requêtes temps réel (PostgreSQL/BigQuery font mieux) |
| Agrégations, jointures, nettoyage à grande échelle | Traitement de petits datasets (pandas plus rapide) |
| Batch ET Streaming (Structured Streaming) | OLTP (opérations transactionnelles) |
| SQL via temp views | Remplacer un orchestrateur (Kestra, Airflow) |

---

## Récapitulatif Final

### Spark — Points clés

- **Calcul distribué** : découpe les données en partitions, traitement parallèle sur N workers
- **Lazy Evaluation** : rien ne s'exécute avant une action (`show`, `count`, `write`)
- **Transformations vs Actions** : transformations = plan, actions = exécution
- **Shuffle** : opération coûteuse à minimiser (`groupBy`, `join`, `orderBy`)
- **Parquet** : format de référence, columnar + compressé
- **Spark UI port 4040** : monitoring des jobs en temps réel

### PySpark — Points clés

- **SparkSession** : point d'entrée unique, `getOrCreate()`
- **PySpark via uv** : standalone, pas besoin de `SPARK_HOME`
- **Java 17** : prérequis obligatoire
- **Temp views** : `createOrReplaceTempView()` pour faire du SQL pur
- **`collect()`** : uniquement sur de petits résultats

### Ta stack DE
```
Infrastructure (Terraform)
    ↓
Containerisation (Docker)
    ↓
Ingestion (dlt)
    ↓
Orchestration (Kestra)
    ↓
Warehouse (BigQuery)
    ↓
Transformation (dbt)
    ↓
Batch Processing (Spark) ← Module 6
    ↓
Streaming (Kafka) ← Module 7
```

---

*Notes du Module 6 - DataTalks Club DE Zoomcamp*